# Brusselator model

In [ ]:
import numpy as np
from scipy.optimize import root
from scipy.integrate import solve_ivp
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
pio.templates.default = "seaborn"
import warnings
warnings.filterwarnings('ignore')

The dynamics of the oscillating reaction discovered by Belousov and Zhabotinsky, can be modeled through the so-called Brusselator model depending on two parameters:

$$
\left\{\begin{aligned}
d_t y_1 & = a - (b+1) y_1 + y_1^2y_2\\
d_t y_2 & = b y_1 - y_1^2y_2
\end{aligned}\right.
$$

In [ ]:
class brusselator_model:

    def __init__(self, a, b):
        self.a = a
        self.b = b

    def fcn(self, t, y):
        y1, y2 = y
        a = self.a
        b = self.b
        y1_dot = a - (b+1)*y1 + y1*y1*y2
        y2_dot = b*y1 - y1*y1*y2
        return np.array([y1_dot, y2_dot])

    def jac(self, t, y):
        y1, y2 = y
        a = self.a
        b = self.b
        return np.array([[2*y1*y2 -(b+1), y1*y1], 
                         [-2*y1*y2 + b, -y1*y1] ])

## Quasi-exact solution

The quasi-exact solution is obtained by using an explicit Runge-Kutta method of order 5 with stepsize control and fine tolerances due to Dormand and Prince.

In [ ]:
bm = brusselator_model(a=1, b=4)
fcn = bm.fcn  

tini = 0. 
tend = 20.

yini = (1.5, 3)

sol = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=1.e-12, atol=1.e-12)

fig = make_subplots(rows=2, cols=2, specs=[[{"colspan": 2}, None],[{}, None]], 
                    subplot_titles=("Solution","Phase plane"), vertical_spacing=0.2)
fig.add_trace(go.Scatter(x=sol.t, y=sol.y[0], name='y1'), row=1, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=sol.y[1], name='y2'), row=1, col=1)

fig.add_trace(go.Scatter(x=sol.y[0], y=sol.y[1], showlegend=False), row=2, col=1)

legend = dict(x=0.9, bgcolor='rgba(0,0,0,0)')
fig.update_layout(legend=legend, height=700)
fig.update_xaxes(title_text='t', row=1)
fig.update_xaxes(title_text='y1', row=2)
fig.update_yaxes(title_text='y2', row=2)

## Characterisation of stiffness

In [ ]:
bm = brusselator_model(a=1, b=4)
fcn = bm.fcn  
jac = bm.jac

tini = 0. 
tend = 20.

yini = (1.5 , 3)

sol = solve_ivp(fcn, (tini, tend), yini, method="RK45", rtol=1.e-12, atol=1.e-12)

eig_vals = np.zeros((sol.t.size, 2), dtype=np.complex128)
for it, yi in enumerate(sol.y.transpose()):
    eig_vals[it] = np.linalg.eigvals(jac(0, yi))

lambda1 = eig_vals[:, 0]
lambda2 = eig_vals[:, 1]

fig = make_subplots(rows=2, cols=1, vertical_spacing=0.15,
                    subplot_titles=("Real part of eigenvalues","Imaginary part of eigenvalues"))
color = fig.layout.template.layout.colorway
fig.add_trace(go.Scatter(x=sol.t, y=np.real(lambda1), name='lambda1', line_color=color[0], legendgroup='real'), row=1, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=np.real(lambda2), name='lambda2', line_color=color[1], legendgroup='real'), row=1, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=np.imag(lambda1), name='lambda1', line_color=color[0], legendgroup='imag'), row=2, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=np.imag(lambda2), name='lambda2', line_color=color[1], legendgroup='imag'), row=2, col=1)
legend = dict(x=0.9, bgcolor='rgba(0,0,0,0)', tracegroupgap=260, groupclick="toggleitem")
fig.update_layout(legend=legend, height=700)
fig.update_xaxes(title_text='t', row=1)
fig.update_xaxes(title_text='t', row=2)

## Runge-Kutta methods

In [ ]:
def sol_and_error_bruss(yini, tini, tend, a, b, nt, method):
    
    if method == rk1:
        str_method = "RK1 (explicit Euler)"
    elif method == rk2:
        str_method = "RK2"
    elif method == rk3:
        str_method = "RK3"
    elif method == rk4:
        str_method = "RK4"
    elif method == implicit_euler:
        str_method = "Implicit Euler"
    else:
        print("Unkown method")
        return
    
    bm = brusselator_model(a, b)
    fcn = bm.fcn

    sol_exa = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=1.e-12, atol=1.e-12)

    specs=[[{"colspan": 2}, None], [{}, {}]]
    fig = make_subplots(rows=2, cols=2, specs=specs, subplot_titles=("Solution", "Phase Plane", "Global error"), vertical_spacing=0.15)
    fig.add_trace(go.Scattergl(x=sol_exa.t, y=sol_exa.y[0], name='y1', showlegend=False, line_color='grey', legendgroup='sol'), row=1, col=1)
    fig.add_trace(go.Scattergl(x=sol_exa.t, y=sol_exa.y[1], name='y2', showlegend=False, line_color='grey', legendgroup='sol'), row=1, col=1)
    fig.add_trace(go.Scattergl(x=sol_exa.y[0], y=sol_exa.y[1], showlegend=False, legendgroup='err', line_color='grey'), row=2, col=1)

    marker_y1 = dict(size=5, symbol='x-thin', line=dict(width=1, color=color[0]))
    line_y1 = dict(dash='dot', color=color[0])
    marker_y2 = dict(size=5, symbol='x-thin', line=dict(width=1, color=color[1]))
    line_y2 = dict(dash='dot', color=color[1])
    marker_pp = dict(size=5, symbol='x-thin', line=dict(width=1, color=color[2]))
    line_pp = dict(dash='dot', color=color[2])


    for nt_i in nt:
        sol = method(tini, tend, nt_i, yini, fcn)
        sol_exa = solve_ivp(fcn, (tini, tend), yini, method="DOP853", t_eval=sol.t, rtol=1.e-12, atol=1.e-12)
        err = np.abs(sol.y-sol_exa.y)

        fig.add_trace(go.Scattergl(visible=False, x=sol.t, y=sol.y[0], name='y1', mode="lines+markers", legendgroup='sol',
                                 marker=marker_y1, line=line_y1), row=1, col=1)
        fig.add_trace(go.Scattergl(visible=False, x=sol.t, y=sol.y[1], name='y2', mode="lines+markers", legendgroup='sol',
                                 marker=marker_y2, line=line_y2), row=1, col=1)

        fig.add_trace(go.Scattergl(visible=False, x=sol.y[0], y=sol.y[1], showlegend=False, mode="lines+markers",
                                 marker=marker_pp, line=line_pp), row=2, col=1)

        fig.add_trace(go.Scattergl(visible=False, x=sol.t, y=err[0], name='y1', mode="markers", legendgroup='err',
                                 marker=marker_y1), row=2, col=2)
        fig.add_trace(go.Scattergl(visible=False, x=sol.t, y=err[1], name='y2', mode="markers", legendgroup='err',
                                 marker=marker_y2), row=2, col=2)

    fig.data[-1].visible = True
    fig.data[-2].visible = True
    fig.data[-3].visible = True
    fig.data[-4].visible = True
    fig.data[-5].visible = True

    steps = []
    for i, nt_i in enumerate(nt):
        args = [{"visible": [(el<3) or ((el>=5*i+3) and (el<5*i+8)) for el in range(len(fig.data))]}]
        step = dict(method="update", label = f"{nt_i}", args=args)
        steps.append(step)
    sliders = [dict(active=(nt.size-1), currentvalue={"prefix": "nt : "}, steps=steps)]

    legend = dict(x=0.9, bgcolor='rgba(0,0,0,0)', tracegroupgap=280, groupclick="toggleitem")
    fig.update_layout(height=750, sliders=sliders, legend=legend)
    fig.show()

### Order 1

In [ ]:
class ode_result:
    def __init__(self, y, t, nfev):
        self.y = y
        self.t = t
        self.nfev = nfev 
        
def rk1(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt))
    y[:,0] = yini

    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        y[:,it+1] = yn + dt*fcn(tn, yn)
        
    nfev = nt-1

    return ode_result(y, t, nfev)

In [ ]:
yini = (1.5 , 3)
tini = 0.
tend = 20.
a = 1.
b = 4.

nt_max = 2000
nt = np.hstack((np.array([233, 234]), np.geomspace(235, nt_max, num=10, dtype=int)))

sol_and_error_bruss(yini, tini, tend, a, b, nt, rk1)

### Order 2

In [ ]:
def rk2(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt))
    y[:,0] = yini

    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        k1 = fcn(tn, yn)
        k2 = fcn(tn + 0.5*dt, yn + dt*(0.5*k1))
        y[:,it+1] = yn + dt*k2

    nfev = 2*(nt-1)

    return ode_result(y, t, nfev)

In [ ]:
yini = (1.5 , 3)
tini = 0.
tend = 20.
a = 1.
b = 4.

nt_max = 2000
nt = np.hstack((np.array([224, 225]), np.geomspace(227, nt_max, num=10, dtype=int)))
sol_and_error_bruss(yini, tini, tend, a, b, nt, rk2)

### Order 3

In [ ]:
def rk3(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt))
    y[:,0] = yini

    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        k1 = fcn(tn, yn)
        k2 = fcn(tn + 0.5*dt, yn + dt*(0.5*k1))
        k3 = fcn(tn + dt, yn + dt*(-k1 + 2*k2))
        y[:,it+1] = yn + (dt/6)*(k1+4*k2+k3)

    nfev = 3*(nt-1)

    return ode_result(y, t, nfev)

In [ ]:
yini = (1.5 , 3)
tini = 0.
tend = 20.
a = 1.
b = 4.

nt_max = 2000
nt = np.hstack((np.array([179, 180, 181]), np.geomspace(183, nt_max, num=10, dtype=int)))

sol_and_error_bruss(yini, tini, tend, a, b, nt, rk3)

### Order 4

In [ ]:
def rk4(tini, tend, nt, yini, fcn):

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt), order='F')
    y[:,0] = yini

    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        k1 = fcn(tn, yn)
        k2 = fcn(tn + 0.5*dt, yn + dt*(0.5*k1))
        k3 = fcn(tn + 0.5*dt, yn + dt*(0.5*k2))
        k4 = fcn(tn + dt, yn + dt*k3)
        y[:,it+1] = yn + (dt/6)*(k1+2*k2+2*k3+k4)

    nfev = 4*(nt-1)

    return ode_result(y, t, nfev)

In [ ]:
yini = (1.5 , 3)
tini = 0.
tend = 20.
a = 1.
b = 4.

nt_max = 2000
nt = np.hstack((np.array([130, 131, 132, 133, 134]), np.geomspace(135, nt_max, num=10, dtype=int)))

sol_and_error_bruss(yini, tini, tend, a, b, nt, rk4)

### Computational cost

In [ ]:
tini = 0.
tend = 20.

yini = (1.5 , 3)

bm = brusselator_model(a=1, b=4)
fcn = bm.fcn  

nt = np.array([1001, 2001, 5001, 10001, 25001, 50001, 100001, 200001])

fe_rk1=[]; norm_rk1=[]
fe_rk2=[]; norm_rk2=[]
fe_rk3=[]; norm_rk3=[]
fe_rk4=[]; norm_rk4=[]

norm_rk11=[]
norm_rk22=[]
norm_rk33=[]
norm_rk44=[]

for nt_i in nt:

    t = np.linspace(tini,tend,nt_i)
    
    rtol=1e-13; atol=1e-16
    sol_exa = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=rtol, atol=atol, t_eval=t)

    sol_rk1 = rk1(tini, tend, nt_i, yini, fcn)
    fe_rk1.append(sol_rk1.nfev)
    norm_rk1.append(np.max(np.abs(sol_exa.y-sol_rk1.y)))

    sol_rk2 = rk2(tini, tend, nt_i, yini, fcn)
    fe_rk2.append(sol_rk2.nfev)
    norm_rk2.append(np.max(np.abs(sol_exa.y-sol_rk2.y)))
    
    sol_rk3 = rk3(tini, tend, nt_i, yini, fcn)
    fe_rk3.append(sol_rk3.nfev)
    norm_rk3.append(np.max(np.abs(sol_exa.y-sol_rk3.y)))
    
    sol_rk4 = rk4(tini, tend, nt_i, yini, fcn)
    fe_rk4.append(sol_rk4.nfev)
    norm_rk4.append(np.max(np.abs(sol_exa.y-sol_rk4.y)))


dt = (tend-tini)/(nt-1)    
    
fig = go.Figure()
fig.add_trace(go.Scatter(x=fe_rk1, y=norm_rk1, name='rk1', marker_symbol='x'))
fig.add_trace(go.Scatter(x=fe_rk2, y=norm_rk2, name='rk2', marker_symbol='x'))
fig.add_trace(go.Scatter(x=fe_rk3, y=norm_rk3, name='rk3', marker_symbol='x'))
fig.add_trace(go.Scatter(x=fe_rk4, y=norm_rk4, name='rk4', marker_symbol='x'))
fig.add_trace(go.Scatter(x=fe_rk4, y=norm_rk4, name='rk4', marker_symbol='x'))
fig.add_trace(go.Scatter(x=fe_rk4, y=np.power(1e-3*np.array(fe_rk4),-4), name='slope -4', mode='lines', line_dash='dot'))
fig.update_xaxes(type='log', exponentformat='e', title="Number of function evaluations")
fig.update_yaxes(type='log', exponentformat='e', title="l-infinite norm")
fig.update_layout(height=500, legend=dict(x=0.9, bgcolor='rgba(0,0,0,0)'))

### Implicit Euler

In [ ]:
def implicit_euler(tini, tend, nt, yini, fcn): 
 
    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini_array = np.array(yini)
    neq = yini_array.size

    y = np.zeros((neq, nt), order='F')
    y[:,0] = yini_array
 
    def g(uip1, *args): 
        uip, tip1 = args 
        return uip1 - uip - dt*fcn(tip1, uip1)
 
    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        y0 = yn + dt*fcn(tn, yn)
        # solve y[:,it+1] - y[:,it] - dt * fcn(tini + (it+1)*dt, y[:,it+1]) = 0 
        sol = root(g, y0, (yn, tn+dt))
        y[:,it+1] = sol.x

    nfev = nt-1
         
    return ode_result(y, t, nfev)

In [ ]:
yini = (1.5 , 3)
tini = 0.
tend = 20.
a = 1.
b = 4.

nt_max = 2000
nt = np.hstack((np.array([40, 50]), np.geomspace(60, nt_max, num=10, dtype=int)))

sol_and_error_bruss(yini, tini, tend, a, b, nt, implicit_euler)

## Embedded Runge-Kutta methods (based on 3/8 fourth-order)

In [ ]:
class embedded_ode_result:

    def __init__(self, y, t, nfev, t_rej, dt, dt_rej, loc_err_est, loc_err):
        self.y = y
        self.t = t
        self.nfev = nfev 
        self.t_rej = t_rej
        self.dt = dt
        self.dt_rej = dt_rej
        self.loc_err_est = loc_err_est
        self.loc_err = loc_err

def rk_embedded(tini, tend, yini, fcn, tol, dt_ini=1.e-2):

    # butcher table of the "3/8 rule" 
    c2 = 1/3; c3 = 2/3; c4 = 1.

    a21 = 1/3; 
    a31 = -1/3; a32 = 1.
    a41 = 1.;   a42 = -1; a43 = 1.

    b1 = 1/8; b2 = 3/8; b3 = 3/8; b4 = 1/8

    # coefficients bhat of the embedded method
    b1hat = 2*b1 - 1/6 
    b2hat = 2*(1-c2)*b2 
    b3hat = 2*(1-c3)*b3
    b4hat = 0.
    b5hat = 1/6
     
    # init all parameters    
    dt = dt_ini

    yini = np.atleast_1d(yini)
    neq = yini.size

    # list
    t_acc = [tini]
    t_rej = []
    dt_acc = []
    dt_rej = []
    y = [yini] 
    loc_err_est = [0.]
    loc_err = [0.]

    tn = tini
    yn = yini

    k5 = np.array(fcn(tn, yn))
    nfev = 1

    while (tn < tend):

        # yn compute with the "3/8 rule" method
        k1 = k5
        k2 = fcn(tn + c2*dt, yn + dt*a21*k1)
        k3 = fcn(tn + c3*dt, yn + dt*(a31*k1 + a32*k2))
        k4 = fcn(tn + c4*dt, yn + dt*(a41*k1 + a42*k2 + a43*k3))

        ynp1 = yn + dt*(b1*k1 + b2*k2 + b3*k3 + b4*k4)

        k5 = np.array(fcn(tn+dt, ynp1))
        nfev += 4

        # yn compute with the embeddded method
        yhatnp1 = yn + dt*(b1hat*k1 + b2hat*k2 + b3hat*k3 + b4hat*k4 + b5hat*k5)

        # compute local error estimation 
        err = np.max(np.abs(ynp1 - yhatnp1)) 

        dtopt = dt * min(5, max(0.2, 0.9*np.power(tol/err, 1./4.)))

        if (err < tol):

            # compute and save exact local error
            sol = solve_ivp(fcn, (tn, tn+dt), yn, method="DOP853", rtol=1.e-12, atol=1.e-12)
            loc_err.append(np.max(np.abs(ynp1 - sol.y[:,-1])))
            
            # update for next step
            tn = tn + dt
            yn = ynp1

            # save time, solution and local error estimation
            t_acc.append(tn)
            dt_acc.append(dt) 
            y.append(yn)
            loc_err_est.append(err)

            # compute next dt
            dt = min(dtopt, tend-tn)

        else:
            t_rej.append(tn)
            dt_rej.append(dt) 
            dt = dtopt
            k5 = k1

    y = np.array(np.transpose(np.array(y)), order='F')

    return embedded_ode_result(y, np.array(t_acc), nfev,  np.array(t_rej), np.array(dt_acc), np.array(dt_rej),
                               np.array(loc_err_est), np.array(loc_err))

In [ ]:
tini = 0.
tend = 20.

yini = (1.5 , 3)

bm = brusselator_model(a=1, b=4)
fcn = bm.fcn  

tol = 1e-2
sol = rk_embedded(tini, tend, yini, fcn, tol)
sol_exa = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=1.e-12, atol=1.e-15, t_eval=sol.t)

glob_err = np.empty(sol.t.size)
for i in range(sol.t.size):
    glob_err[i] = np.sqrt(1./2) * np.linalg.norm(sol_exa.y[:,i] - sol.y[:,i])


print("   Number of time steps : " + str(sol.t.size-1))
print("   Number of function evaluations : " + str(sol.nfev))

fig = make_subplots(rows=3, cols=1, vertical_spacing=0.15,
                   subplot_titles=(f"Solution", f"Time steps ", f"The different errors, tol={tol}"))

color = fig.layout.template.layout.colorway
marker_y1 = dict(size=5, symbol='x-thin', line=dict(width=1, color=color[0]))
line_y1 = dict(color=color[0])
marker_y2 = dict(size=5, symbol='x-thin', line=dict(width=1, color=color[1]))
line_y2 = dict(color=color[1])

marker_3 = dict(size=5, symbol='square', color=color[6])

fig.add_trace(go.Scatter(x=sol.t, y=sol.y[0], name='y1', mode='lines+markers', marker=marker_y1, line=line_y1, 
              legendgroup='sol'), row=1, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=sol.y[1], name='y2', mode='lines+markers', marker=marker_y2, line=line_y2, 
              legendgroup='sol'), row=1, col=1)

fig.add_trace(go.Scatter(x=sol.t[:-1], y=sol.dt, name='accepted dt', mode='markers', marker_symbol='x', 
                         legendgroup='dt'), row=2, col=1)
fig.add_trace(go.Scatter(x=sol.t_rej, y=sol.dt_rej, name='rejected dt', mode='markers', marker_symbol='square', 
                         legendgroup='dt'), row=2, col=1)


fig.add_trace(go.Scatter(x=sol.t, y=sol.loc_err_est, name='loc. error est.', mode='markers', 
                         marker_symbol='x', legendgroup='err'), row=3, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=sol.loc_err, name='loc. error', mode='markers', marker_symbol='square', 
                         legendgroup='err'), row=3, col=1)
fig.add_trace(go.Scatter(x=sol.t, y=glob_err, name='glob. error', mode='markers', marker_symbol='circle', 
                         legendgroup='err'), row=3, col=1)
fig.add_hline(y=tol, line_width=2, line_dash="dot", row=3, col=1)

fig.update_layout(legend=dict(tracegroupgap=275), height=1000)
fig.update_xaxes(title='t', col=1)
fig.update_yaxes(type='log', exponentformat='e', title='dt', row=2)
fig.update_yaxes(type='log', exponentformat='e', row=3, range=[np.log10(tol)-4, np.log10(np.max(np.array([tol,np.max(glob_err)])))+1])
fig.show()

### Computational cost

In [ ]:
def rk38(tini, tend, nt, yini, fcn):
    
    # butcher table of the "3/8 rule" 
    c2 = 1/3; c3 = 2/3; c4 = 1.

    a21 = 1/3; 
    a31 = -1/3; a32 = 1.
    a41 = 1.;   a42 = -1; a43 = 1.

    b1 = 1/8; b2 = 3/8; b3 = 3/8; b4 = 1/8

    # coefficients bhat of the embedded method
    b1hat = 2*b1 - 1/6 
    b2hat = 2*(1-c2)*b2 
    b3hat = 2*(1-c3)*b3
    b4hat = 0.
    b5hat = 1/6

    dt = (tend-tini) / (nt-1)
    t = np.linspace(tini, tend, nt)

    yini = np.atleast_1d(yini)
    neq = yini.size

    y = np.zeros((neq, nt), order='F')
    y[:,0] = yini
    
    for it, tn  in enumerate(t[:-1]):
        yn = y[:,it]
        k1 = fcn(tn, yn)
        k2 = fcn(tn + c2*dt, yn + dt*a21*k1)
        k3 = fcn(tn + c3*dt, yn + dt*(a31*k1 + a32*k2))
        k4 = fcn(tn + c4*dt, yn + dt*(a41*k1 + a42*k2 + a43*k3))
        y[:,it+1] = yn + dt*(b1*k1 + b2*k2 + b3*k3 + b4*k4)

    nfev = 4*(nt-1)

    return ode_result(y, t, nfev)

In [ ]:
tini = 0.
tend = 20.

yini = (1.5 , 3)

bm = brusselator_model(a=1, b=4)
fcn = bm.fcn  

nt = [1001, 2001, 5001, 10001, 20001, 50001]
fe_rk38=[]; norm_rk38=[]

for nt_i in nt:
    t = np.linspace(tini,tend,nt_i)
    rtol=1e-13; atol=1e-16
    sol_exa = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=rtol, atol=atol, t_eval=t)
    sol_rk38 = rk38(tini, tend, nt_i, yini, fcn)
    fe_rk38.append(sol_rk38.nfev)
    #norm_rk38.append(np.linalg.norm(sol_exa.y-sol_rk38.y) / np.sqrt(nt_i))
    norm_rk38.append(np.max(np.abs(sol_exa.y-sol_rk38.y)))
    
tol = [1.e-6, 1.e-8, 1.e-10, 1e-12, 1e-14]
fe_rk_emb=[]; norm_rk_emb=[]

for tol_i in tol:
    sol_rk_emb = rk_embedded(tini, tend, yini, fcn, tol_i)
    rtol=1e-13; atol=1e-16
    sol_exa = solve_ivp(fcn, (tini, tend), yini, method="DOP853", rtol=rtol, atol=atol, t_eval=sol_rk_emb.t)
    fe_rk_emb.append(sol_rk_emb.nfev)
    norm_rk_emb.append(np.max(np.abs(sol_exa.y-sol_rk_emb.y)))

    
fig = go.Figure()
fig.add_trace(go.Scatter(x=fe_rk38, y=norm_rk38, name='rk4 with fixed dt', marker_symbol='x'))
fig.add_trace(go.Scatter(x=fe_rk_emb, y=norm_rk_emb, name='rk4 with adaptative dt', marker_symbol='x'))
fig.update_xaxes(type='log', exponentformat='e', title="Number of function evaluations")
fig.update_yaxes(type='log', exponentformat='e', title="error (l-infinite norm)")
fig.update_layout(height=500)
fig.show()